# Expected VE for activities other than running

`calculate-ve.ipynb` predicts VE from running pace. Badminton, Muay Thai, cycling or a park gym session have no
pace, so this notebook goes through oxygen uptake instead:

```
activity text  ->  hazewise.db (2024 Compendium of Physical Activities)  ->  MET
MET            ->  VO2 (mL/min) = MET x 3.5 x body weight (kg)
VO2 + profile  ->  VE, from a model fitted on the Málaga treadmill tests (VO2 and VE measured breath by breath)
```

Running still goes through `calculate-ve.ipynb`, which uses pace and running history and is the better model for
runs. This notebook covers everything else. `assess-risk.ipynb` and the web app pick between the two.

In the web app a language model (Gemini) turns free text such as "doubles badminton for an hour and a half" into an
activity name, an intensity and a duration. The database lookup in §2 is what decides the MET; the language model
never supplies a number that goes into the dose.

## 1. Inputs

In [1]:
ACTIVITY = {"activity": "badminton", "intensity": "moderate", "minutes": 90}   # intensity: light, moderate or vigorous
PERSON = {"age": 30, "sex": "male", "height_cm": 175, "weight_kg": 70}

## 2. Activity lookup

`hazewise.db` holds the 1,111 Compendium activities with their MET values, plus 18 activity families (running,
badminton, martial arts, ...) that each name a light, moderate and vigorous Compendium code, and 115 aliases that map
everyday words to a family ("muay thai", "fitness corner", "pcn").

The lookup tries, in order:

1. **alias**: the longest alias that appears as whole words in the text, e.g. "muay thai" → martial_arts.
2. **family name**: the text names a family directly.
3. **full-text search** over the Compendium descriptions (SQLite FTS5, porter stemming), exercise headings first.
   Words about the session ("class", "with friends", "90 min") are dropped and every remaining word must match, so
   "underwater basket weaving" finds nothing rather than basketball. This finds activities without a family, such as
   "rock climbing" or "frisbee".

A family gives the code for the chosen intensity. A full-text hit gives that one code, whatever the intensity. No
match returns `None`, "activity not found", and the app then shows the forecast without a dose.

In [2]:
import re
import sqlite3
from pathlib import Path

DB = Path("hazewise.db")
con = sqlite3.connect(DB)
con.row_factory = sqlite3.Row
INTENSITIES = ("light", "moderate", "vigorous")
FAMILIES = {r["family"]: dict(r) for r in con.execute("select * from activity_families")}
ALIASES = dict(con.execute("select alias, family from family_aliases").fetchall())
ACT_COLS = "code, major_heading, met, name, effort, speed_kmh_min, speed_kmh_max, indoor_only, exercise_relevant"


def norm(text):
    return " ".join(re.sub(r"[^a-z0-9]+", " ", str(text).lower()).split())


def activity_by_code(code):
    r = con.execute(f"select {ACT_COLS} from activities where code = ?", (code,)).fetchone()
    return dict(r) if r else None


def find_family(text):
    t = f" {norm(text)} "
    hits = [(len(a), a, f) for a, f in ALIASES.items() if f" {norm(a)} " in t]
    hits += [(len(f), f, f) for f in FAMILIES if f" {norm(f)} " in t]
    return max(hits)[2] if hits else None


# words that describe the session rather than the activity; dropped before full-text search
STOPWORDS = set((
    "a an and at for from go going do doing done i in into my of on or our play playing some the to with we "
    "class classes session sessions training practice workout morning evening afternoon night today tonight tomorrow "
    "weekend friends friend group easy light moderate hard vigorous intense casual competitive min mins minute minutes "
    "hour hours hr hrs km k").split())


def search_terms(text):
    return [w for w in norm(text).split() if len(w) > 1 and w not in STOPWORDS and not w.isdigit()]


def fts_search(text, limit=5):
    tokens = search_terms(text)
    if not tokens:
        return []
    q = " ".join(f'"{w}"' for w in tokens)               # every term must match
    rows = con.execute(
        f"select {', '.join('a.' + c.strip() for c in ACT_COLS.split(','))} from activities_fts f "
        "join activities a on a.code = f.code where activities_fts match ? "
        "order by a.exercise_relevant desc, f.rank, a.code limit ?", (q, limit)).fetchall()
    return [dict(r) for r in rows]


def resolve_activity(text, intensity="moderate"):
    intensity = intensity if intensity in INTENSITIES else "moderate"
    fam = find_family(text)
    if fam:
        f = FAMILIES[fam]
        a = activity_by_code(f[f"{intensity}_code"])
        return {**a, "family": fam, "family_name": f["display_name"], "intensity": intensity, "matched_by": "family"}
    hits = fts_search(text)
    if hits:
        return {**hits[0], "family": None, "family_name": None, "intensity": None, "matched_by": "search"}
    return None


for q in ["badminton", "Muay Thai class", "cycling along the PCN", "park gym", "fitness corner workout",
          "rock climbing", "ultimate frisbee", "underwater basket weaving"]:
    r = resolve_activity(q)
    print(f"{q!r:28} -> " + (f"{r['code']} {r['met']:>4} MET  {r['name'][:60]}  ({r['matched_by']}"
                              + (f", {r['family']})" if r["family"] else ")") if r else "activity not found"))

'badminton'                  -> 15020  7.0 MET  Badminton, competitive  (family, badminton)
'Muay Thai class'            -> 15430 10.3 MET  Martial Arts, different types, moderate pace (e.g., judo, ju  (family, martial_arts)
'cycling along the PCN'      -> 01016  7.0 MET  Bicycling, self-selected moderate pace  (family, cycling)
'park gym'                   -> 02052  5.0 MET  Resistance (weight) training, squats, deadlift, slow or expl  (family, weight_training)
'fitness corner workout'     -> 02052  5.0 MET  Resistance (weight) training, squats, deadlift, slow or expl  (family, weight_training)
'rock climbing'              -> 15535  7.3 MET  Rock climbing, ascending rock, high difficulty  (search)
'ultimate frisbee'           -> 15250  8.0 MET  Frisbee, ultimate  (search)
'underwater basket weaving'  -> activity not found


## 3. Data

The same cleaning and 10 s medians as `calculate-ve.ipynb` §2, keeping VO2 this time. Breaths with VO2 or VCO2 at or
below zero lose their gas values, as in `explore.ipynb` §2. The modelling set is every exercise bin (ramp, plus steady
walking after the first minute of warm-up) with VO2 above 200 mL/min, VE and a full profile.

In [3]:
import json

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold

DATA = Path("data/treadmill-maximal-exercise-tests-from-the-exercise-physiology-and-human-performance-lab-of-the-university-of-malaga-1.0.1")
pd.set_option("display.precision", 2)
pd.set_option("display.width", 140)

meas = pd.read_csv(DATA / "test_measure.csv")
subj = pd.read_csv(DATA / "subject-info.csv")

m = meas.sort_values(["ID_test", "time"]).reset_index(drop=True)
m.loc[(m.VO2 <= 0) | (m.VCO2 <= 0), ["VO2", "VCO2"]] = np.nan
g = m.groupby("ID_test")
ve_med = g.VE.transform(lambda s: s.rolling(7, center=True, min_periods=3).median())
m = m[~((m.VE - ve_med).abs() > 0.5 * ve_med)].copy()

g = m.groupby("ID_test")
pk = g.Speed.transform("max")
recovery = (m.Speed == pk).groupby(m.ID_test).cummax() & (m.Speed < pk)
moving = m.Speed > 0
t_move0 = m.time.where(moving).groupby(m.ID_test).transform("min")
warm_speed = m.Speed.where(moving & (m.time < t_move0 + 60)).groupby(m.ID_test).transform("median")
t_ramp0 = m.time.where(~recovery & (m.Speed <= warm_speed + 0.05)).groupby(m.ID_test).transform("max")
m["phase"] = np.select([recovery, ~recovery & (m.time > t_ramp0), ~recovery & moving, ~recovery & ~moving],
                       ["recovery", "ramp", "warmup", "rest"], default="other")
m["t_phase"] = m.time - m.groupby(["ID_test", "phase"]).time.transform("min")

m["tbin"] = (m.time // 10).astype(int)
bins = (m.groupby(["ID_test", "ID", "phase", "tbin"])[["time", "t_phase", "Speed", "VO2", "VE"]].median().reset_index()
        .merge(subj.drop(columns="ID"), on="ID_test", how="left"))
ex = bins[(bins.phase == "ramp") | ((bins.phase == "warmup") & (bins.t_phase >= 60))]
dv = ex.dropna(subset=["VO2", "VE", "Age", "Sex", "Weight", "Height"])
dv = dv[dv.VO2 > 200].reset_index(drop=True)
dv["VO2kg"] = dv.VO2 / dv.Weight
yv = dv.VE.to_numpy()
print(f"{len(dv):,} bins from {dv.ID_test.nunique()} tests / {dv.ID.nunique()} subjects; "
      f"VO2 {dv.VO2kg.quantile(.01):.1f} to {dv.VO2kg.quantile(.99):.1f} mL/kg/min (1st-99th percentile)")

81,508 bins from 981 tests / 846 subjects; VO2 11.6 to 59.1 mL/kg/min (1st-99th percentile)


## 4. VO2 → VE model

Ordinary least squares on log VE, as in `calculate-ve.ipynb` §3. VO2 enters as log VO2 (L/min), plus hinges
`max(0, VO2/kg − k)` at 20, 30, 40 and 50 mL/kg/min, so VE can climb faster once breathing passes the ventilatory
thresholds. Age, sex, log weight and height enter linearly. Predictions go back to litres with Duan smearing.

Cross-validated by subject, 5 folds. The first row has no hinges, the second is the model, and the last row is the
running model A from `calculate-ve.ipynb` on the same folds, which predicts from speed instead of VO2.

In [4]:
VO2_KNOTS = [20, 30, 40, 50]
SPEED_KNOTS = [6.5, 8, 10, 13, 16]


def design_vo2(df, knots=VO2_KNOTS):
    X = pd.DataFrame({"logVO2": np.log(df.VO2 / 1000)}, index=df.index)
    for k in knots:
        X[f"max(0,VO2kg-{k})"] = (df.VO2 / df.Weight - k).clip(lower=0)
    X["Age"], X["Sex"], X["logW"], X["Height"] = df.Age, df.Sex, np.log(df.Weight), df.Height
    return X


def design_speed(df):
    X = pd.DataFrame({"Speed": df.Speed}, index=df.index)
    for k in SPEED_KNOTS:
        X[f"max(0,Speed-{k})"] = (df.Speed - k).clip(lower=0)
    X["Age"], X["Sex"], X["logW"], X["Height"] = df.Age, df.Sex, np.log(df.Weight), df.Height
    return X


class VO2Formula:
    def __init__(self, df, make=design_vo2):
        self.make = make
        X, t = make(df), np.log(df.VE)
        self.ols = LinearRegression().fit(X, t)
        self.smear = np.mean(np.exp(t - self.ols.predict(X)))

    def predict(self, df):
        return np.exp(self.ols.predict(self.make(df))) * self.smear

    def coefs(self):
        return pd.Series(np.r_[self.ols.intercept_, self.ols.coef_, self.smear],
                         index=["intercept", *self.ols.feature_names_in_, "smear"])


def scores(yv, p):
    ape = np.abs(p - yv) / yv
    return {"MAE L/min": np.mean(np.abs(p - yv)), "median APE %": 100 * np.median(ape),
            "90th pct APE %": 100 * np.quantile(ape, 0.9), "R²": 1 - np.sum((p - yv) ** 2) / np.sum((yv - yv.mean()) ** 2)}


folds = list(GroupKFold(n_splits=5).split(dv, groups=dv.ID))
makers = {"log VO2 only": lambda df: design_vo2(df, []), "log VO2 + hinges (model)": design_vo2, "speed (running model A)": design_speed}
oof = {k: np.empty(len(dv)) for k in makers}
for tr, te in folds:
    for k, mk in makers.items():
        oof[k][te] = VO2Formula(dv.iloc[tr], mk).predict(dv.iloc[te])
display(pd.DataFrame({k: scores(yv, p) for k, p in oof.items()}).T)

,MAE L/min,median APE %,90th pct APE %,R²
log VO2 only,8.20,10.63,27.06,0.86
log VO2 + hinges (model),7.93,10.16,26.03,0.87
speed (running model A),8.61,11.87,31.01,0.86


Measured VO2 predicts VE a little better than speed does, which is expected: VO2 already carries the person's running
economy. The hinges help modestly. These scores assume VO2 is known exactly. For a real activity it comes from a MET
value, which is the bigger source of error, and §5 measures that.

## 5. Test: MET instead of measured VO2

The lab only has running and walking, but the Compendium has METs for running at set speeds, so the whole chain can
be tested on the treadmill data: speed → Compendium MET → MET × 3.5 × weight → model → VE, against the measured VE.

* Compendium METs are steady-state values, and the ramp lags steady state by 0.55 km/h (`calculate-ve.ipynb` §6). A
  ramp bin at speed s is compared with the MET for s − 0.55.
* The MET curve uses the Compendium's level running codes (4.3 to 14 mph), linear between their speeds.
* Folds are the same as §4, so the model never sees the person it predicts.

In [5]:
LAB_RAMP_LAG_KMH = 0.55
LEVEL_RUN_CODES = ["12028", "12029", "12030", "12045", "12050", "12060", "12070", "12080", "12090", "12100", "12110",
                   "12115", "12120", "12130", "12132", "12134", "12135"]
lv = (pd.read_sql(f"select code, met, speed_kmh_min, speed_kmh_max from activities where code in "
                  f"({','.join('?' * len(LEVEL_RUN_CODES))})", con, params=LEVEL_RUN_CODES)
      .assign(speed=lambda r: (r.speed_kmh_min + r.speed_kmh_max) / 2).sort_values("speed"))

r = dv[dv.phase == "ramp"].copy()
r["steady"] = r.Speed - LAB_RAMP_LAG_KMH
r = r[r.steady.between(lv.speed.min(), lv.speed.max())].copy()
r["met"] = np.interp(r.steady, lv.speed, lv.met)
r["vo2_met"] = r.met * 3.5 * r.Weight
vo2_ratio = (r.VO2 / r.vo2_met).groupby(r.ID_test).median()
print(f"{len(r):,} ramp bins, {r.ID_test.nunique()} tests")
print("measured VO2 ÷ MET VO2, per test: " +
      ", ".join(f"p{int(q * 100)} {v:.2f}" for q, v in vo2_ratio.quantile([.1, .5, .9]).items()))

chain, known = np.empty(len(r)), np.empty(len(r))
for tr, te in folds:
    f = VO2Formula(dv.iloc[tr])
    mask = r.ID.isin(set(dv.ID.iloc[te])).to_numpy()
    chain[mask] = f.predict(r[mask].assign(VO2=r.vo2_met[mask]))
    known[mask] = f.predict(r[mask])
display(pd.DataFrame({"measured VO2": scores(r.VE.to_numpy(), known),
                      "MET → VO2 (full chain)": scores(r.VE.to_numpy(), chain)}).T)

54,140 ramp bins, 980 tests
measured VO2 ÷ MET VO2, per test: p10 0.84, p50 0.97, p90 1.15


,MAE L/min,median APE %,90th pct APE %,R²
measured VO2,10.39,11.00,28.00,0.75
MET → VO2 (full chain),11.47,12.21,33.41,0.70


For running, the Compendium MET is close to the lab's measured VO2 on average, and the person-to-person spread is
around ±15%. Feeding MET instead of measured VO2 adds roughly a percentage point of median error. The full chain lands
near the speed model, so the MET route is a fair substitute when there is no pace.

Running is the best case. Its METs come from many measured studies at fixed speeds. A sport's MET averages play and
rest over whoever was measured, and the session's real intensity varies much more. §6 widens the band for that.

## 6. Error band

As in `calculate-ve.ipynb` §7, the band comes from per-test error: the median of actual ÷ predicted across each test's
bins, then the 10th and 90th percentiles across tests.

* **gait** (Compendium headings Running and Walking): the full-chain error from §5 as measured.
* **sport** (everything else): the gait error plus an assumed extra MET spread of ±25% at the 10th/90th percentile,
  log-normal and independent. The 25% is a judgement, not a measurement. It stands for session intensity varying
  around the Compendium average, which the lab can't test.

In [6]:
SPORT_MET_P90 = 1.25
q = [0.1, 0.5, 0.9]
gait_logr = np.log((r.VE / chain).groupby(r.ID_test.values).median())
rng = np.random.default_rng(0)
draws = rng.choice(gait_logr.to_numpy(), 200_000) + rng.normal(0, np.log(SPORT_MET_P90) / 1.2816, 200_000)
VO2_BAND = pd.DataFrame({"gait": np.exp(gait_logr.quantile(q)).to_numpy(), "sport": np.exp(np.quantile(draws, q))},
                    index=["p10", "p50", "p90"]).T
display(VO2_BAND.round(3))

,p10,p50,p90
gait,0.80,0.96,1.20
sport,0.72,0.97,1.35


## 7. Final model

Refit on every bin. log VE = intercept + Σ coefficient × term, VE = exp(that) × smear.

In [7]:
VO2_MODEL = VO2Formula(dv)
display(VO2_MODEL.coefs().to_frame("coefficient").map(lambda c: f"{c:.4g}"))
VO2_RANGE = {"VO2 mL/kg/min": (dv.VO2kg.quantile(0.01), dv.VO2kg.quantile(0.99)), "age": (dv.Age.min(), dv.Age.max()),
         "weight kg": (dv.Weight.min(), dv.Weight.max()), "height cm": (dv.Height.min(), dv.Height.max())}
print("valid range:", ", ".join(f"{k} {lo:.4g}-{hi:.4g}" for k, (lo, hi) in VO2_RANGE.items()))

,coefficient
intercept,2.862
logVO2,0.8851
"max(0,VO2kg-20)",0.01194
"max(0,VO2kg-30)",0.0008217
"max(0,VO2kg-40)",0.000575
"max(0,VO2kg-50)",-0.01168
Age,0.00254
Sex,0.1184
logW,0.0663
Height,-0.0002265


valid range: VO2 mL/kg/min 11.64-59.1, age 10.8-63, weight kg 41-135, height cm 150-203


## 8. Estimate

`estimate_ve_activity` returns the same fields as `estimate_ve` in `calculate-ve.ipynb`, so `assess-risk.ipynb` can use
either one. The activity is assumed to hold its MET for the whole session.

In [8]:
GAIT_HEADINGS = {"Running", "Walking"}


def estimate_ve_activity(activity, person):
    a = activity if "met" in activity else resolve_activity(activity["activity"], activity.get("intensity", "moderate"))
    if a is None:
        raise LookupError(f"activity not found: {activity.get("activity")!r}")
    minutes = float(activity["minutes"])
    vo2 = a["met"] * 3.5 * person["weight_kg"]            # mL/min
    x = pd.DataFrame({"VO2": [vo2], "Age": person["age"], "Sex": int(person["sex"] == "female"),
                      "Weight": person["weight_kg"], "Height": person["height_cm"]})
    ve = VO2_MODEL.predict(x)[0]
    kind = "gait" if a["major_heading"] in GAIT_HEADINGS else "sport"
    band = VO2_BAND.loc[kind]
    notes = [f"{a['name']}: {a['met']:g} MET (Compendium {a['code']})",
             f"VO2 {vo2:.0f} mL/min = {a['met']:g} MET × 3.5 × {person['weight_kg']:g} kg"]
    warn = []
    if a["indoor_only"]:
        warn.append("this activity is usually done indoors; outdoor PM2.5 overstates the dose")
    if a["major_heading"] == "Sports":
        warn.append("the MET averages play and rest; hard rallies or rounds breathe more than the average")
    checks = {"VO2 mL/kg/min": vo2 / person["weight_kg"], "age": person["age"], "weight kg": person["weight_kg"],
              "height cm": person["height_cm"]}
    for k, v in checks.items():
        lo, hi = VO2_RANGE[k]
        if not lo <= v <= hi:
            warn.append(f"{k} {v:.4g} outside the data ({lo:.4g}-{hi:.4g})")
    return {"variant": "MET", "band_kind": kind, "code": a["code"], "met": a["met"], "vo2_mlmin": vo2,
            "minutes": minutes, "ve_lmin": ve, "ve_p10": ve * band.p10, "ve_p90": ve * band.p90,
            "air_m3": ve * minutes / 1000, "air_p10_m3": ve * band.p10 * minutes / 1000,
            "air_p90_m3": ve * band.p90 * minutes / 1000, "notes": notes, "warnings": warn}


r_ = estimate_ve_activity(ACTIVITY, PERSON)
print(f"Plan: {ACTIVITY["activity"]}, {ACTIVITY['intensity']}, {ACTIVITY['minutes']} min")
print("".join(f"  · {n}\n" for n in r_["notes"]))
print(f"Expected VE          {r_['ve_lmin']:.1f} L/min   (80% band {r_['ve_p10']:.1f}-{r_['ve_p90']:.1f}, {r_['band_kind']})")
print(f"Air over the session {r_['air_m3']:.2f} m³      (80% band {r_['air_p10_m3']:.2f}-{r_['air_p90_m3']:.2f})")
for w in r_["warnings"]:
    print(f"  ! {w}")

Plan: badminton, moderate, 90 min
  · Badminton, competitive: 7 MET (Compendium 15020)
  · VO2 1715 mL/min = 7 MET × 3.5 × 70 kg

Expected VE          41.5 L/min   (80% band 29.8-55.9, sport)
Air over the session 3.73 m³      (80% band 2.68-5.03)
  ! the MET averages play and rest; hard rallies or rounds breathe more than the average


Some activities at each intensity, for the same 70 kg man over an hour:

In [9]:
rows = []
for text in ["badminton", "muay thai", "cycling", "park gym", "football", "tai chi", "walking", "running"]:
    for inten in INTENSITIES:
        a = resolve_activity(text, inten)
        v = estimate_ve_activity({**a, "minutes": 60}, PERSON)
        rows.append({"activity": text, "intensity": inten, "MET": a["met"], "VO2 L/min": v["vo2_mlmin"] / 1000,
                     "VE L/min": v["ve_lmin"], "p10": v["ve_p10"], "p90": v["ve_p90"], "air m³/h": v["air_m3"]})
display(pd.DataFrame(rows).set_index(["activity", "intensity"]).round(2))

MET  VO2 L/min  VE L/min    p10     p90  air m³/h
activity  intensity                                                    
badminton light       5.5       1.35     31.77  22.82   42.80      1.91
          moderate    7.0       1.72     41.50  29.80   55.91      2.49
          vigorous    9.0       2.20     56.42  40.52   76.01      3.39
muay thai light       5.3       1.30     30.74  22.08   41.42      1.84
          moderate   10.3       2.52     67.38  48.39   90.78      4.04
          vigorous   14.3       3.50    108.27  77.76  145.87      6.50
cycling   light       4.0       0.98     23.96  17.21   32.29      1.44
          moderate    7.0       1.72     41.50  29.80   55.91      2.49
          vigorous    9.0       2.20     56.42  40.52   76.01      3.39
park gym  light       3.5       0.86     21.29  15.29   28.69      1.28
          moderate    5.0       1.23     29.20  20.97   39.34      1.75
          vigorous    6.0       1.47     34.72  24.94   46.78      2.08
football  light       3.5       0.86     21.29  15.29   28.69      1.28
          moderate    7.0       1.72     41.50  29.80   55.91      2.49
          vigorous    9.5       2.33     60.52  43.47   81.54      3.63
tai chi   light       1.5       0.37     10.06   7.22   13.55      0.60
          moderate    3.3       0.81     20.21  14.52   27.23      1.21
          vigorous    6.0       1.47     34.72  24.94   46.78      2.08
walking   light       2.8       0.69     17.48  13.89   21.01      1.05
          moderate    3.8       0.93     22.90  18.20   27.52      1.37
          vigorous    4.8       1.18     28.16  22.38   33.85      1.69
running   light       7.5       1.84     45.04  35.79   54.14      2.70
          moderate    9.3       2.28     58.87  46.78   70.75      3.53
          vigorous   11.8       2.89     81.32  64.63   97.74      4.88

Running appears for comparison only. The app sends runs to `calculate-ve.ipynb`, which can use the planned pace and
running history.

## 9. Export for the web app

`web/src/lib/ve-others-model.json` holds the coefficients, band and ranges. `web/src/lib/__tests__/ve-others.golden.json`
holds results from the functions above for the TypeScript port to match, including activity lookups.

In [10]:
WEB = Path("web/src/lib")
c = VO2_MODEL.coefs()
model_json = {
    "source": "calculate-ve-others.ipynb (refit on all bins, §7)",
    "vo2Knots": VO2_KNOTS,
    "intercept": float(c["intercept"]),
    "coef": {k: float(v) for k, v in c.drop(["intercept", "smear"]).items()},
    "smear": float(c["smear"]),
    "band": {k: {"p10": float(VO2_BAND.loc[k, "p10"]), "p90": float(VO2_BAND.loc[k, "p90"])} for k in VO2_BAND.index},
    "gaitHeadings": sorted(GAIT_HEADINGS),
    "range": {k: [float(lo), float(hi)] for k, (lo, hi) in VO2_RANGE.items()},
}
(WEB / "ve-others-model.json").write_text(json.dumps(model_json, indent=1, ensure_ascii=False) + "\n", encoding="utf-8")

people = [PERSON, {"age": 68, "sex": "female", "height_cm": 155, "weight_kg": 52},
          {"age": 16, "sex": "male", "height_cm": 170, "weight_kg": 140}]
cases = []
for text, inten, mins in [("badminton", "moderate", 90), ("Muay Thai class", "vigorous", 60), ("cycling along the pcn", "light", 45),
                          ("park gym", "moderate", 40), ("brisk walk", "vigorous", 30), ("tai chi", "light", 60),
                          ("rock climbing", "moderate", 120), ("treadmill", "moderate", 30)]:
    for p in people:
        a = resolve_activity(text, inten)
        v = estimate_ve_activity({**a, "minutes": mins}, p)
        cases.append({"text": text, "intensity": inten, "minutes": mins, "person": p, "code": a["code"],
                      "out": {k: v[k] for k in ["variant", "band_kind", "met", "vo2_mlmin", "minutes", "ve_lmin", "ve_p10",
                                                "ve_p90", "air_m3", "notes", "warnings"]}})
lookups = []
for text in ["badminton", "Badminton doubles with friends", "muay thai", "MUAY-THAI!!", "cycling along the PCN", "park gym",
             "fitness corner workout", "outdoor gym", "hiit", "zumba in the park", "frisbee", "ultimate frisbee",
             "rock climbing", "skateboarding", "kayaking", "swimming laps", "underwater basket weaving", "", "walk the dog"]:
    out = {}
    for inten in INTENSITIES:
        a = resolve_activity(text, inten)
        out[inten] = None if a is None else {"code": a["code"], "family": a["family"], "matched_by": a["matched_by"]}
    lookups.append({"text": text, "out": out})
(WEB / "__tests__" / "ve-others.golden.json").write_text(
    json.dumps({"cases": cases, "lookups": lookups}, indent=1, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"wrote {len(cases)} VE cases and {len(lookups)} lookups")

wrote 24 VE cases and 19 lookups


## 10. Limits

* **MET is an average.** Intermittent sports swing between rest and bursts, and VE rises faster than VO2 at high
  effort, so the average VE of a badminton match is above VE at the average VO2. Estimates for stop-start sports lean
  low.
* **The sport band is partly assumed** (§6). Only the running chain is measured.
* **One MET per session.** Warm-up, breaks and cool-down aren't modelled; the intensity choice should describe the
  session as a whole.
* **MET uses 3.5 mL/kg/min per MET**, the Compendium's convention. Resting VO2 is lower in older and heavier people,
  so the 2024 Compendium suggests corrected METs for them. That correction isn't applied here.
* **Lab conditions.** Same as `calculate-ve.ipynb`: 15 to 32 °C, short tests, ages 11 to 63. Heat and long sessions
  likely push VE above these estimates.